In [1]:
import ee

# =========================================================
# 1. AUTH / INIT
# =========================================================
# Replace with your real Google Cloud project ID
ee.Initialize()

# =========================================================
# 2. SETTINGS
# =========================================================
YEAR = 2022
N_POINTS = 1000
SEED = 42
SCALE_METERS = 9000
EXPORT_DESCRIPTION = f"us_wind_speed_dataset_{YEAR}"
EXPORT_FILENAME = f"us_wind_speed_dataset_{YEAR}"

# Contiguous U.S. bounding box
conus_bbox = ee.Geometry.Rectangle([-125.0, 24.0, -66.5, 49.5], geodesic=False)

# =========================================================
# 3. RANDOM SAMPLE POINTS
# =========================================================
points = ee.FeatureCollection.randomPoints(
    region=conus_bbox,
    points=N_POINTS,
    seed=SEED,
    maxError=1000
)

def add_latlon(feature):
    coords = feature.geometry().coordinates()
    return feature.set({
        "lon": coords.get(0),
        "lat": coords.get(1),
        "year": YEAR
    })

points = points.map(add_latlon)

# =========================================================
# 4. LOAD ERA5-LAND WIND COMPONENTS
# =========================================================
wind_ic = (
    ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")
    .filterDate(f"{YEAR}-01-01", f"{YEAR+1}-01-01")
    .select([
        "u_component_of_wind_10m",
        "v_component_of_wind_10m"
    ])
)

# =========================================================
# 5. COMPUTE WIND SPEED = sqrt(u^2 + v^2)
# =========================================================
def add_wind_speed(image):
    u = image.select("u_component_of_wind_10m")
    v = image.select("v_component_of_wind_10m")
    wind_speed = u.pow(2).add(v.pow(2)).sqrt().rename("wind_speed")
    return image.addBands(wind_speed)

wind_with_speed = wind_ic.map(add_wind_speed)

# =========================================================
# 6. YEARLY AVERAGE WIND SPEED
# =========================================================
annual_mean_wind = wind_with_speed.select("wind_speed").mean().rename("annual_mean_wind_speed")

# =========================================================
# 7. SAMPLE YEARLY MEAN WIND AT RANDOM POINTS
# =========================================================
samples = annual_mean_wind.sampleRegions(
    collection=points,
    properties=["lat", "lon", "year"],
    scale=SCALE_METERS,
    geometries=False
)

# =========================================================
# 8. EXPORT TO GOOGLE DRIVE AS CSV
# =========================================================
task = ee.batch.Export.table.toDrive(
    collection=samples,
    description=EXPORT_DESCRIPTION,
    fileNamePrefix=EXPORT_FILENAME,
    fileFormat="CSV"
)

task.start()
print(f"Started export task: {EXPORT_DESCRIPTION}")

Started export task: us_wind_speed_dataset_2022


In [5]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

# =========================================================
# 1. LOAD DATA
# =========================================================
df = pd.read_csv("wind_with_elevation.csv")

# =========================================================
# 2. CLEAN DATA
# =========================================================
df = df.dropna()

# Ensure required columns exist
required_cols = ["lat", "lon", "annual_mean_wind_speed"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing column: {col}")

# Remove invalid values
df = df[df["annual_mean_wind_speed"] > 0]

# =========================================================
# 3. FEATURES / TARGET
# =========================================================
# Use elevation if available (recommended)
features = ["lat", "lon"]
if "elevation" in df.columns:
    features.append("elevation")

X = df[features]
y = df["annual_mean_wind_speed"]

# =========================================================
# 4. TRAIN / TEST SPLIT
# =========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================================================
# 5. TRAIN MODEL (XGBOOST)
# =========================================================
model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

model.fit(X_train, y_train)

# =========================================================
# 6. EVALUATE
# =========================================================
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Model performance:")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R^2:  {r2:.4f}")

# =========================================================
# 7. TEST PREDICTION
# =========================================================
sample = pd.DataFrame([{
    "lat": 39.74,
    "lon": -104.99,
    **({"elevation": 1600} if "elevation" in features else {})
}])

prediction = model.predict(sample)[0]
print(f"\nPredicted annual wind speed: {prediction:.4f}")

# =========================================================
# 8. SAVE MODEL
# =========================================================
joblib.dump(model, "wind_xgboost_model.pkl")
print("\nModel saved as wind_xgboost_model.pkl")

Model performance:
MAE:  0.2562
RMSE: 0.3874
R^2:  0.8801

Predicted annual wind speed: 2.7166

Model saved as wind_xgboost_model.pkl


In [3]:
import ee

# =========================================================
# 1. AUTH / INIT
# =========================================================

ee.Initialize()


# =========================================================
# 2. SETTINGS
# =========================================================
YEAR = 2022
N_POINTS = 10000
SEED = 42
SCALE = 9000

# US bounding box
region = ee.Geometry.Rectangle([-125, 24, -66, 49])

# =========================================================
# 3. RANDOM POINTS
# =========================================================
points = ee.FeatureCollection.randomPoints(
    region=region,
    points=N_POINTS,
    seed=SEED
)

def add_latlon(f):
    coords = f.geometry().coordinates()
    return f.set({
        "lon": coords.get(0),
        "lat": coords.get(1),
        "year": YEAR
    })

points = points.map(add_latlon)

# =========================================================
# 4. WIND DATA (ERA5)
# =========================================================
wind_ic = (
    ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")
    .filterDate(f"{YEAR}-01-01", f"{YEAR+1}-01-01")
    .select(["u_component_of_wind_10m", "v_component_of_wind_10m"])
)

def add_wind_speed(img):
    u = img.select("u_component_of_wind_10m")
    v = img.select("v_component_of_wind_10m")
    ws = u.pow(2).add(v.pow(2)).sqrt().rename("wind_speed")
    return img.addBands(ws)

wind_ic = wind_ic.map(add_wind_speed)

annual_wind = wind_ic.select("wind_speed").mean()

# =========================================================
# 5. ELEVATION
# =========================================================
elevation = ee.Image("USGS/SRTMGL1_003").rename("elevation")

# =========================================================
# 6. COMBINE FEATURES
# =========================================================
combined = annual_wind.rename("annual_mean_wind_speed").addBands(elevation)

# =========================================================
# 7. SAMPLE DATA
# =========================================================
samples = combined.sampleRegions(
    collection=points,
    properties=["lat", "lon", "year"],
    scale=SCALE,
    geometries=False
)

# =========================================================
# 8. EXPORT
# =========================================================
task = ee.batch.Export.table.toDrive(
    collection=samples,
    description="wind_with_elevation",
    fileNamePrefix="wind_with_elevation",
    fileFormat="CSV"
)

task.start()

print("Export started 🚀")

Export started 🚀
